In [0]:
from pyspark.sql.functions import to_date, col, when, year, month, dayofmonth, dayofweek, date_format
from datetime import datetime
import requests
import csv
import time
import json

In [0]:
# Authenticating Azure Data Lake Storage Gen2
storage_account = "capitaledgestorageacct"

# get storage access key
storage_key = "azO2UKU4KCiR7J1rvy3QhGcu6WbOGDc1rqXn4aFC0HA5J+DODfKL/dWAibXVhKjyg6NqBYlGcj99+AStE3iwkg=="

# configure databricks to access storage account
spark.conf.set(
  f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
  storage_key
)

SILVER = f"abfss://silver-capitaledge@{storage_account}.dfs.core.windows.net/"
GOLD = f"abfss://gold-capitaledge@{storage_account}.dfs.core.windows.net/"
DATA_TRACKING = f"abfss://capitaledge-data-tracking@{storage_account}.dfs.core.windows.net/"

In [0]:
df = spark.read.parquet(f'{SILVER}silver_capitaledge/').orderBy(col("date").desc())
display(df)

In [0]:
# Getting the last stock date from latest stock date ADLS
last_date_df = spark.read.option("multiline", "true").json(f'{DATA_TRACKING}/gold_last_date.json')

# Getting the actual last date value
last_date = last_date_df.where(col("1-pipeline") == "gold").select("2-latest_date").first()[0]

# Getting the actual ealier date value
#### early_date = last_date_df.where(col("1-pipeline") == "gold").select("3-earlier_date").first()[0]

# Converting to date type
last_date = datetime.strptime(last_date, "%Y-%m-%d").date()
#### early_date = datetime.strptime(early_date, "%Y-%m-%d").date()

print(last_date)
#### print(early_date)

In [0]:
df_filtered = df.filter(col("date") > last_date) 
display(df_filtered)

In [0]:
# Adding additional columns for business analytics
df_filtered = df_filtered \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("day", dayofmonth(col("date"))) \
    .withColumn("day_of_week", dayofweek(col("date"))) \
    .withColumn("day_name", date_format(col("date"), "EEEE"))

display(df_filtered)

In [0]:
# Getting the latest stock date to update our watermark in ADLS
new_last_date = df_filtered.orderBy(col("date").desc()).select("date").first()[0]

# Updating and overwriting the latest stock date to ADLS
gold_last_date = {
    "1-pipeline": "gold",
    "2-latest_date": new_last_date,
    "3-earlier_date": last_date
}

file_path = f"{DATA_TRACKING}/gold_last_date.json"

# Save the JSON data
json_data = json.dumps(gold_last_date, indent=4, default=str)
dbutils.fs.put(file_path, json_data, overwrite=True)

In [0]:
# Save the business analysis ready DataFrame to the Gold container
gold_output_path = f"{GOLD}gold_capitaledge/"

# Append DataFrame to Gold container in Parquet format
df_filtered.write.mode('append').parquet(gold_output_path)